In [ ]:
from telethon import TelegramClient
from telethon.errors import FloodWaitError
import json, os, asyncio
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv()

API_ID   = int(os.getenv('API_ID'))
API_HASH = os.getenv('API_HASH')

GROUPS = [
    'joinhackingarmy',
    'HackingBlogsGroup',
    'cloudandcybersecurity',
    'cybdetective',
    'cissp',
    'PHOfficial',
    'WokeIntelDrops',
    'itsectalk',
    'hackers_asylum',
    'cybersecurityexperts',
]

EXPECTED = {
    'joinhackingarmy': 716, 'HackingBlogsGroup': 791,
    'cloudandcybersecurity': 2151, 'cybdetective': 3019,
    'cissp': 7492, 'PHOfficial': 7732, 'WokeIntelDrops': 13152,
    'itsectalk': 46386, 'hackers_asylum': 1750,
    'cybersecurityexperts': 233226,
}

START_DATE = datetime(2023, 1, 1, tzinfo=timezone.utc)
END_DATE   = datetime(2025, 6, 30, tzinfo=timezone.utc)

OUTPUT_DIR = r'sentinel_replica_jsons'
os.makedirs(OUTPUT_DIR, exist_ok=True)

client = TelegramClient('session_sentinel', API_ID, API_HASH)

async def coletar_grupo(group):
    filepath = os.path.join(OUTPUT_DIR, f'{group}.json')

    # Carrega o que já existe
    messages = []
    if os.path.exists(filepath):
        with open(filepath, encoding='utf-8') as f:
            messages = json.load(f)

    threshold = EXPECTED.get(group, 0) * 0.9
    if len(messages) >= threshold:
        print(f"Pulando {group} — {len(messages)} msgs já coletadas")
        return

    print(f"Coletando {group} — já tem {len(messages)} msgs, continuando...")

    collected_ids = {m.get('_id') for m in messages if '_id' in m}

    concluido = False
    while not concluido:
        try:
            async for msg in client.iter_messages(group, offset_date=END_DATE, reverse=False):
                if msg.date < START_DATE:
                    break
                if msg.text:
                    entry = {'_id': msg.id, 'date': msg.date.strftime('%Y-%m-%d'), 'message': msg.text}
                    if msg.id not in collected_ids:
                        messages.append(entry)
                        collected_ids.add(msg.id)
                    if len(messages) % 1000 == 0:
                        with open(filepath, 'w', encoding='utf-8') as f:
                            json.dump(messages, f, ensure_ascii=False, indent=2)
                        print(f"  {group}: {len(messages)} msgs...")
            concluido = True  # só marca como concluído se terminar sem erro

        except FloodWaitError as e:
            print(f"  FloodWait em {group}: aguardando {e.seconds}s...")
            await asyncio.sleep(e.seconds + 5)
            print(f"  Retomando {group}...")
            # loop while repete a coleta

        except Exception as e:
            print(f"  Erro em {group}: {e}")
            concluido = True  # erros não-FloodWait encerram

        finally:
            clean = [{'date': m['date'], 'message': m['message']} for m in messages]
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump(clean, f, ensure_ascii=False, indent=2)
            print(f"  -> {group}: {len(clean)} msgs salvas")

async def coletar_tudo():
    await client.start()
    for group in GROUPS:
        await coletar_grupo(group)
    await client.disconnect()

await coletar_tudo()